# Project: Tomographic reconstruction with Huber regularization (2D tests)

We want to solve
$$ \min_x \frac{1}{2} \| A(x) - y \|_2^2 + \lambda H_\epsilon(\nabla x) $$
with steepest descent. The method is first tested in 2D on the Shepp-Logan phantom.
The functions shared with the 3D notebook are in `utils.py`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import odl
from odl.applications import tomo

from utils import save_figure, show_images, add_noise, relative_error, huber_reconstruction, parker_weights

ModuleNotFoundError: No module named 'utils'

## 2D setup

In [ ]:
reco_space_2d = odl.uniform_discr([-112, -112], [112, 112], (128, 128), dtype='float32')
geometry_2d = tomo.cone_beam_geometry(
    space=reco_space_2d,
    src_radius=780.0,
    det_radius=220.0,
    num_angles=128,
    short_scan=True,
)
ray_trafo_2d = tomo.RayTransform(vol_space=reco_space_2d, geometry=geometry_2d)
data_space_2d = ray_trafo_2d.range

phantom = odl.phantom.shepp_logan(reco_space_2d, modified=False)
exact_data = ray_trafo_2d(phantom)

In [ ]:
show_images([phantom], ['Phantom (full range)'], clim=(0, 2), filename='phantom_full')
show_images([phantom], ['Phantom (narrow window)'], filename='phantom_window')

In [ ]:
noisy_data, delta = add_noise(exact_data, 0.01)
print(f'Noise level delta = {delta:.2f}, relative noise = {delta / exact_data.norm():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(exact_data.asarray(), aspect='auto')
axes[0].set_title('Exact data')
axes[1].imshow(noisy_data.asarray(), aspect='auto')
axes[1].set_title('Noisy data (1% noise)')
save_figure('data_2d')
plt.show()

## Solver

The gradient of the objective is Lipschitz with constant $L = \|A\|^2 + \lambda \|\nabla\|^2 / \epsilon$,
so steepest descent with constant step $t < 2/L$ converges. We use $t = 1.9/L$ (see `huber_reconstruction` in `utils.py`).
The operator norms are only estimated once since this is slow in 3D.

In [ ]:
ray_trafo_norm_2d = float(ray_trafo_2d.norm(estimate=True))
grad_norm_2d = float(odl.Gradient(reco_space_2d).norm(estimate=True))
op_norms_2d = (ray_trafo_norm_2d, grad_norm_2d)
print(f'||A|| = {ray_trafo_norm_2d:.2f}, ||grad|| = {grad_norm_2d:.2f}')

### First test

In [ ]:
niter = 3000

x, errors = huber_reconstruction(ray_trafo_2d, noisy_data, lam=1.0, eps=0.01, niter=niter,
                                 op_norms=op_norms_2d, x_true=phantom)

show_images([phantom, x], ['Phantom', f'Huber, error = {errors[-1]:.3f}'], filename='first_test_reco')

plt.figure()
plt.plot(errors)
plt.xlabel('Iteration')
plt.ylabel('Relative error')
plt.yscale('log')
plt.grid(True)
plt.title(r'Relative error, $\lambda = 1$, $\epsilon = 0.01$')
save_figure('first_test_error')
plt.show()

## Choice of $\epsilon$

$\lambda$ is fixed and we compare different smoothing parameters $\epsilon$.

In [ ]:
eps_values = [1e-3, 1e-2, 1e-1, 1.0]

eps_recos = []
plt.figure()
for eps in eps_values:
    x, errors = huber_reconstruction(ray_trafo_2d, noisy_data, lam=1.0, eps=eps, niter=niter,
                                     op_norms=op_norms_2d, x_true=phantom)
    eps_recos.append(x)
    plt.plot(errors, label=rf'$\epsilon = {eps}$')
    print(f'eps = {eps}: final error = {errors[-1]:.4f}')

plt.xlabel('Iteration')
plt.ylabel('Relative error')
plt.yscale('log')
plt.grid(True)
plt.title(r'Relative error for different $\epsilon$ ($\lambda = 1$)')
plt.legend()
save_figure('eps_error')
plt.show()

show_images(eps_recos, [rf'$\epsilon = {eps}$' for eps in eps_values], filename='eps_recos')

In [ ]:
eps = 1e-3  # chosen from the experiment above

## Choice of $\lambda$: Morozov's discrepancy principle

For a decreasing sequence $\lambda_1 > \lambda_2 > \dots$ we pick the first $\lambda_i$ such that
$\| A x_{\lambda_i} - y \| \le \tau \delta$ with $\tau > 1$.

In [ ]:
tau = 1.1
lam_values = np.logspace(1, -2, 10)

residuals = []
final_errors = []
recos = []
for lam in lam_values:
    x, errors = huber_reconstruction(ray_trafo_2d, noisy_data, lam, eps, niter, op_norms_2d, x_true=phantom)
    recos.append(x)
    residuals.append((ray_trafo_2d(x) - noisy_data).norm())
    final_errors.append(errors[-1])
    print(f'lambda = {lam:.4f}: residual / delta = {residuals[-1] / delta:.3f}, error = {errors[-1]:.4f}')

# first (largest) lambda that satisfies the discrepancy principle
for i_morozov, res in enumerate(residuals):
    if res <= tau * delta:
        break
else:
    print('No lambda satisfies the discrepancy principle, using the smallest one')
lam_morozov = lam_values[i_morozov]
print(f'Morozov: lambda = {lam_morozov:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].loglog(lam_values, residuals, 'o-', label='residual')
axes[0].axhline(tau * delta, color='r', linestyle='--', label=r'$\tau \delta$')
axes[0].axvline(lam_morozov, color='k', linestyle=':', label=r'Morozov $\lambda$')
axes[0].set_xlabel(r'$\lambda$')
axes[0].set_ylabel(r'$\|Ax - y\|$')
axes[0].grid(True)
axes[0].legend()

axes[1].semilogx(lam_values, final_errors, 'o-')
axes[1].axvline(lam_morozov, color='k', linestyle=':', label=r'Morozov $\lambda$')
axes[1].set_xlabel(r'$\lambda$')
axes[1].set_ylabel('Relative error')
axes[1].grid(True)
axes[1].legend()
save_figure('morozov_2d')
plt.show()

x_morozov = recos[i_morozov]
show_images([phantom, x_morozov], ['Phantom', rf'$\lambda = {lam_morozov:.3f}$, error = {final_errors[i_morozov]:.3f}'],
            filename='morozov_2d_reco')

## Error vs iterations for different $\lambda$ and noise levels

In [ ]:
noise_levels = [0.01, 0.03, 0.05]
lam_test = [0.1, 1.0, 10.0]

fig, axes = plt.subplots(1, len(noise_levels), figsize=(15, 4), sharey=True)
for ax, noise in zip(axes, noise_levels):
    noisy, _ = add_noise(exact_data, noise)
    for lam in lam_test:
        _, errors = huber_reconstruction(ray_trafo_2d, noisy, lam, eps, niter, op_norms_2d, x_true=phantom)
        ax.plot(errors, label=rf'$\lambda = {lam}$')
    ax.set_title(f'{noise * 100:.0f}% noise')
    ax.set_xlabel('Iteration')
    ax.set_yscale('log')
    ax.grid(True)
    ax.legend()
axes[0].set_ylabel('Relative error')
save_figure('error_vs_iterations')
plt.show()

## Filtered back-projection

The short scan covers $180°$ plus the fan angle, so some lines are measured twice and others once.
FBP assumes every line is counted equally, which gives a shading over the image. Parker weighting fixes
this by weighting each ray so that the weights of two rays measuring the same line sum to 1.

In [ ]:
parker = parker_weights(ray_trafo_2d)

### Check on noise-free data: short scan vs full 360° scan

In [ ]:
fbp_short = tomo.fbp_op(ray_trafo_2d)

geometry_full = tomo.cone_beam_geometry(
    space=reco_space_2d,
    src_radius=780.0,
    det_radius=220.0,
    num_angles=128,
    short_scan=False,
)
ray_trafo_full = tomo.RayTransform(vol_space=reco_space_2d, geometry=geometry_full)
x_full = tomo.fbp_op(ray_trafo_full)(ray_trafo_full(phantom))

images = [
    fbp_short(exact_data),
    fbp_short(tomo.parker_weighting(ray_trafo_2d) * exact_data),
    fbp_short(parker * exact_data),
    x_full,
]
titles = ['Short scan, no weighting', 'Short scan, ODL Parker', 'Short scan, flipped Parker', 'Full 360° scan']
show_images(images, [f'{t}\nerror = {relative_error(x, phantom):.3f}' for t, x in zip(titles, images)],
            filename='parker_check')

### Filter types and frequency scaling (noisy data, with Parker weighting)

In [ ]:
scalings = [1.0, 0.5, 0.25]

for filter_type in ['Ram-Lak', 'Hann']:
    images, titles = [], []
    for scaling in scalings:
        fbp = tomo.fbp_op(ray_trafo_2d, filter_type=filter_type, frequency_scaling=scaling)
        x_fbp = fbp(parker * noisy_data)
        images.append(x_fbp)
        titles.append(f'{filter_type}, scaling = {scaling}\nerror = {relative_error(x_fbp, phantom):.3f}')
    show_images(images, titles, filename=f'fbp_2d_{filter_type}')

In [ ]:
x_fbp = tomo.fbp_op(ray_trafo_2d, filter_type='Hann', frequency_scaling=0.5)(parker * noisy_data)
show_images([phantom, x_fbp, x_morozov],
            ['Phantom',
             f'FBP (Hann, 0.5), error = {relative_error(x_fbp, phantom):.3f}',
             f'Huber, error = {relative_error(x_morozov, phantom):.3f}'],
            filename='comparison_2d')